# **main**

In [ ]:
import os
import pandas as pd
import numpy as np
import scipy.io
from scipy.signal import butter, filtfilt, find_peaks, detrend

# Function to read a MATLAB file
def read_mat_file(file_path):
    try:
        mat_data = scipy.io.loadmat(file_path)
        print(f"Keys in the MATLAB file: {mat_data.keys()}")  # Print keys to understand the structure

        timestamps = mat_data['t'].flatten()
        eog_horizontal = mat_data['A'].flatten()  # Adjust if 'A' is not the correct key
        eog_vertical = mat_data['B'].flatten()    # Adjust if 'B' is not the correct key

        return pd.DataFrame({'timestamp': timestamps, 'eog_horizontal': eog_horizontal, 'eog_vertical': eog_vertical})
    except KeyError as e:
        print(f"KeyError: {e} in file {file_path}")
        return None
    except Exception as e:
        print(f"Unexpected error reading {file_path}: {e}")
        return None

# Filter the EOG signal to remove high-frequency noise
def butter_lowpass(cutoff, fs, order=5):
    nyquist = 0.5 * fs
    normal_cutoff = cutoff / nyquist
    b, a = butter(order, normal_cutoff, btype='low', analog=False)
    return b, a

def lowpass_filter(data, cutoff=1.0, fs=1000.0, order=5):
    b, a = butter_lowpass(cutoff, fs, order=order)
    y = filtfilt(b, a, data)
    return y

# Normalize the signal
def normalize(data):
    return (data - np.mean(data)) / np.std(data)

# Detection functions
def detect_blinks(data, threshold=2.5):  # Adjust threshold based on data
    positive_peaks, _ = find_peaks(data, height=threshold)
    negative_peaks, _ = find_peaks(-data, height=threshold)
    blinks = np.sort(np.concatenate((positive_peaks, negative_peaks)))
    return blinks

def detect_saccades(data, threshold=1.0):  # Adjust threshold based on data
    velocity = np.abs(np.diff(data))
    saccades = np.where(velocity > threshold)[0]
    return saccades

def detect_fixations(data, threshold=0.2, min_duration=100):  # Adjust threshold and min_duration based on data
    velocity = np.abs(np.diff(data))
    below_threshold = velocity < threshold
    fixations = []
    start = None
    for i, val in enumerate(below_threshold):
        if val and start is None:
            start = i
        elif not val and start is not None:
            if i - start >= min_duration:
                fixations.append((start, i))
            start = None
    return fixations

# Define the function to process each participant's data
def process_participant_data(file_path):
    data = read_mat_file(file_path)
    if data is None or data.empty:
        return None

    try:
        fs = 1000  # Sample rate
        participant_id = os.path.splitext(os.path.basename(file_path))[0]

        data['filtered_eog_horizontal'] = lowpass_filter(data['eog_horizontal'], fs=fs)
        data['filtered_eog_vertical'] = lowpass_filter(data['eog_vertical'], fs=fs)

        data['detrended_eog_horizontal'] = detrend(data['filtered_eog_horizontal'])
        data['detrended_eog_vertical'] = detrend(data['filtered_eog_vertical'])

        # Calculate mean, variance, standard deviation, and covariance before normalization
        mean_horizontal = np.mean(data['detrended_eog_horizontal'])
        mean_vertical = np.mean(data['detrended_eog_vertical'])
        variance_horizontal = np.var(data['detrended_eog_horizontal'])
        variance_vertical = np.var(data['detrended_eog_vertical'])
        std_horizontal = np.std(data['detrended_eog_horizontal'])
        std_vertical = np.std(data['detrended_eog_vertical'])
        covariance = np.cov(data['detrended_eog_horizontal'], data['detrended_eog_vertical'])[0, 1]

        data['normalized_eog_horizontal'] = normalize(data['detrended_eog_horizontal'])
        data['normalized_eog_vertical'] = normalize(data['detrended_eog_vertical'])

        blink_threshold = 2.5  # Minimum peak height to consider a blink
        saccade_threshold = 1.0  # Minimum change rate to consider a saccade
        fixation_threshold = 0.2  # Maximum change rate to consider a fixation
        min_fixation_duration = 100  # Minimum samples to consider a fixation (adjust as needed)

        blinks = detect_blinks(data['normalized_eog_horizontal'], threshold=blink_threshold)
        saccades = detect_saccades(data['normalized_eog_horizontal'], threshold=saccade_threshold)
        fixations = detect_fixations(data['normalized_eog_horizontal'], threshold=fixation_threshold, min_duration=min_fixation_duration)

        # Calculate durations in seconds
        blink_durations = [(data['timestamp'][b], data['timestamp'][b + 1]) for b in blinks]
        saccade_durations = [(data['timestamp'][s], data['timestamp'][s + 1]) for s in saccades]
        fixation_durations = [(data['timestamp'][start], data['timestamp'][end]) for start, end in fixations]

        return {
            'mean_signal_horizontal': mean_horizontal,
            'mean_signal_vertical': mean_vertical,
            'variance_horizontal': variance_horizontal,
            'variance_vertical': variance_vertical,
            'std_horizontal': std_horizontal,
            'std_vertical': std_vertical,
            'covariance': covariance,
            'blink_durations': blink_durations,
            'saccade_durations': saccade_durations,
            'fixation_durations': fixation_durations,
            'num_blinks': len(blinks),
            'num_saccades': len(saccades),
            'num_fixations': len(fixations)
        }
    except Exception as e:
        print(f"Unexpected error processing {file_path}: {e}")
        return None

# Main processing loop
def main():
    from google.colab import drive
    drive.mount('/content/drive')

    input_folder = input("Please enter the path to the Google Drive folder containing the MATLAB files: ")
    output_file_path = input("Please enter the path to save the output Excel file: ")

    participant_data_list = []

    files = sorted([f for f in os.listdir(input_folder) if f.endswith('.mat')])

    # Ensure all participant IDs are initialized in participant_data_list
    participant_ids = [f.split('.')[0] for f in files]
    for participant_id in participant_ids:
        participant_data_list.append({
            'participant_id': participant_id,
            'mean_signal_horizontal': np.nan,
            'mean_signal_vertical': np.nan,
            'variance_horizontal': np.nan,
            'variance_vertical': np.nan,
            'std_horizontal': np.nan,
            'std_vertical': np.nan,
            'covariance': np.nan,
            'num_blinks': np.nan,
            'num_saccades': np.nan,
            'num_fixations': np.nan,
            'blink_durations': [],
            'saccade_durations': [],
            'fixation_durations': []
        })

    # Process each file and update the participant_data_list
    for filename in files:
        file_path = os.path.join(input_folder, filename)
        participant_id = filename.split('.')[0]
        print(f"Processing {participant_id}...")
        participant_data = process_participant_data(file_path)

        if participant_data is not None:
            for entry in participant_data_list:
                if entry['participant_id'] == participant_id:
                    entry['mean_signal_horizontal'] = participant_data['mean_signal_horizontal']
                    entry['mean_signal_vertical'] = participant_data['mean_signal_vertical']
                    entry['variance_horizontal'] = participant_data['variance_horizontal']
                    entry['variance_vertical'] = participant_data['variance_vertical']
                    entry['std_horizontal'] = participant_data['std_horizontal']
                    entry['std_vertical'] = participant_data['std_vertical']
                    entry['covariance'] = participant_data['covariance']
                    entry['num_blinks'] = participant_data['num_blinks']
                    entry['num_saccades'] = participant_data['num_saccades']
                    entry['num_fixations'] = participant_data['num_fixations']
                    entry['blink_durations'] = participant_data['blink_durations']
                    entry['saccade_durations'] = participant_data['saccade_durations']
                    entry['fixation_durations'] = participant_data['fixation_durations']
                    break

    # Create DataFrame with durations split into multiple columns
    rows = []
    for data in participant_data_list:
        if not data['blink_durations'] and not data['saccade_durations'] and not data['fixation_durations']:
            rows.append({
                'participant_id': data['participant_id'],
                'mean_signal_horizontal': data['mean_signal_horizontal'],
                'mean_signal_vertical': data['mean_signal_vertical'],
                'variance_horizontal': data['variance_horizontal'],
                'variance_vertical': data['variance_vertical'],
                'std_horizontal': data['std_horizontal'],
                'std_vertical': data['std_vertical'],
                'covariance': data['covariance'],
                'num_blinks': data['num_blinks'],
                'num_saccades': data['num_saccades'],
                'num_fixations': data['num_fixations'],
                'blink_start': np.nan,
                'blink_end': np.nan,
                'saccade_start': np.nan,
                'saccade_end': np.nan,
                'fixation_start': np.nan,
                'fixation_end': np.nan
            })
        else:
            for blink_start, blink_end in data['blink_durations']:
                rows.append({
                    'participant_id': data['participant_id'],
                    'mean_signal_horizontal': data['mean_signal_horizontal'],
                    'mean_signal_vertical': data['mean_signal_vertical'],
                    'variance_horizontal': data['variance_horizontal'],
                    'variance_vertical': data['variance_vertical'],
                    'std_horizontal': data['std_horizontal'],
                    'std_vertical': data['std_vertical'],
                    'covariance': data['covariance'],
                    'num_blinks': data['num_blinks'],
                    'num_saccades': data['num_saccades'],
                    'num_fixations': data['num_fixations'],
                    'blink_start': blink_start,
                    'blink_end': blink_end,
                    'saccade_start': np.nan,
                    'saccade_end': np.nan,
                    'fixation_start': np.nan,
                    'fixation_end': np.nan
                })
            for saccade_start, saccade_end in data['saccade_durations']:
                rows.append({
                    'participant_id': data['participant_id'],
                    'mean_signal_horizontal': data['mean_signal_horizontal'],
                    'mean_signal_vertical': data['mean_signal_vertical'],
                    'variance_horizontal': data['variance_horizontal'],
                    'variance_vertical': data['variance_vertical'],
                    'std_horizontal': data['std_horizontal'],
                    'std_vertical': data['std_vertical'],
                    'covariance': data['covariance'],
                    'num_blinks': data['num_blinks'],
                    'num_saccades': data['num_saccades'],
                    'num_fixations': data['num_fixations'],
                    'blink_start': np.nan,
                    'blink_end': np.nan,
                    'saccade_start': saccade_start,
                    'saccade_end': saccade_end,
                    'fixation_start': np.nan,
                    'fixation_end': np.nan
                })
            for fixation_start, fixation_end in data['fixation_durations']:
                rows.append({
                    'participant_id': data['participant_id'],
                    'mean_signal_horizontal': data['mean_signal_horizontal'],
                    'mean_signal_vertical': data['mean_signal_vertical'],
                    'variance_horizontal': data['variance_horizontal'],
                    'variance_vertical': data['variance_vertical'],
                    'std_horizontal': data['std_horizontal'],
                    'std_vertical': data['std_vertical'],
                    'covariance': data['covariance'],
                    'num_blinks': data['num_blinks'],
                    'num_saccades': data['num_saccades'],
                    'num_fixations': data['num_fixations'],
                    'blink_start': np.nan,
                    'blink_end': np.nan,
                    'saccade_start': np.nan,
                    'saccade_end': np.nan,
                    'fixation_start': fixation_start,
                    'fixation_end': fixation_end
                })

    if rows:
        df = pd.DataFrame(rows)
        df = df.sort_values(by='participant_id')  # Ensure the dataframe is sorted by participant_id
        df.to_excel(output_file_path, index=False)
        print(f"Processed data has been saved to {output_file_path}")
    else:
        print("No data was processed successfully.")

if __name__ == '__main__':
    main()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Please enter the path to the Google Drive folder containing the MATLAB files: /content/drive/MyDrive/participants data
Please enter the path to save the output Excel file: /content/drive/MyDrive/processed8_data.xlsx
Processing 01...
Keys in the MATLAB file: dict_keys(['__header__', '__version__', '__globals__', 'A', 'B', 't'])
Processing 02...
Keys in the MATLAB file: dict_keys(['__header__', '__version__', '__globals__', 'A', 'B', 't'])
Processing 03...
Keys in the MATLAB file: dict_keys(['__header__', '__version__', '__globals__', 'A', 'B', 't'])
Processing 04...
Keys in the MATLAB file: dict_keys(['__header__', '__version__', '__globals__', 'A', 'B', 't'])
Processing 05...
Keys in the MATLAB file: dict_keys(['__header__', '__version__', '__globals__', 'A', 'B', 't'])
Processing 06...
Keys in the MATLAB file: dict_keys(['__header__', '__version__', '__globa

##mahdi

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_squared_error, r2_score

# Load the dataset
file_path = 'Dataset (4).xlsx'
df = pd.read_excel(file_path, sheet_name='Sheet1')

# Features and target columns
feature_columns = [
    'Gender', 'Age', 'Education', 'mean_A', 'mean_B', 'variance_A', 'variance_B', 'std_dev_A', 'std_dev_B',
    'covariance', 'skewness_A', 'skewness_B', 'kurtosis_A', 'kurtosis_B', 'rms_A', 'rms_B', 'max_A', 'max_B',
    'min_A', 'min_B', 'peak_to_peak_A', 'peak_to_peak_B', 'energy_A', 'energy_B', 'zero_crossing_rate_A',
    'zero_crossing_rate_B', 'Number of Blinks', 'Number of Saccades_A', 'Number of Fixations_A', 'blink_start', 'blink_end'
]

target_columns = [
    'MMSE score', 'Navigation and orientation (WQ)', 'Distance estimation (WQ)', 'Spatial anxiety (WQ)',
    'Landmark (VT)', 'Location Egocentric (VT)', 'Path survey (VT)', 'Location Allocentric (VT)', 'Path route (VT)'
]

# Splitting the data into features and target
X = df[feature_columns]
y = df[target_columns]

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Standardizing the features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Initializing and training the KNN model
knn = KNeighborsRegressor(n_neighbors=5)
knn.fit(X_train_scaled, y_train)

# Making predictions
y_pred = knn.predict(X_test)

# Evaluating the model
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f'Mean Squared Error: {mse}')
print(f'R-squared Score: {r2}')


KeyError: "['zero_crossing_rate_A', 'zero_crossing_rate_B'] not in index"